# M3L4 E03 — Debugging con traces
### Modulo 3 · Lecture 4 · Construccion, pruebas y trazabilidad de agentes en produccion

---

## Que necesitas saber antes

| Modulo | Concepto | Por que lo necesitas aca |
|---|---|---|
| M3L4 E00 | Trace, Span, jerarquia | Los 5 casos de falla usan la estructura trace-span que viste en E00 |
| M3L4 E01-E02 | MiniTracer, duration_ms, metadata | Los campos que revisa `diagnose_trace()` son los mismos que construiste |
| M3L1 | Tool contracts | Un span con `output: None` es como una tool que no devuelve contrato |
| M3L2 | Agentes con estado | El loop de agentes (Caso 4) ocurre cuando el estado no avanza |
| Python | `dict.get()`, `Counter` de `collections` | Los usas para inspeccionar traces de forma robusta |

Si no entendes la estructura de un trace, repasa E00 antes de continuar.

---

## Definiciones clave

| Concepto | Definicion simple | Como aparece en este notebook |
|---|---|---|
| **Diagnostico** | Proceso de identificar automaticamente el tipo de falla en una traza | `diagnose_trace()` retorna `{'problem': 'misclassification', ...}` |
| **Misclassification** | El router clasifica el intent incorrectamente | `metadata.expected_intent != metadata.actual_intent` |
| **Retrieval vacio** | La busqueda de documentos no encuentra resultados | Span de retrieval con `output['count'] == 0` o `output['documents'] == []` |
| **Latencia alta** | Un paso del sistema tarda mas de lo esperado | `duration_ms > 3000` en algun span |
| **Loop de agentes** | El mismo agente aparece ejecutandose repetidamente | Mismo `name` de span aparece mas de 2 veces |
| **Error silencioso** | Un span termina con `output: None` sin marcar error | Span con `output == None` |
| **suggested_fix** | Mensaje de accion correctiva sugerida | `'Revisar reglas de routing para diferenciar HR de Finance'` |

---

## Como encaja esto en un sistema de agentes

```
E02: Sistema multi-agente con tracing
    |  Cada request produce un trace con spans
    v
E03: Diagnosticar automaticamente fallas en esos traces
    |  diagnose_trace(trace) -> problem, details, suggested_fix
    v
E04: Golden datasets para medir accuracy del router
    |  Prevencion: detectar misclassification antes de producirla
    v
E11-E12: Alertas, dashboards y mejora continua
    |  Monitoreo automatico en produccion
```

**Objetivo del ejercicio:** aprender a leer trazas para detectar y diagnosticar fallas tipicas en sistemas de agentes.

## Instalacion e imports

Este ejercicio solo necesita la biblioteca estandar de Python:

| Import | Que hace | Por que lo necesitamos |
|---|---|---|
| `import json` | (Opcional) Formatear traces como JSON para inspeccion visual | Para debuggear traces manualmente si es necesario |

`diagnose_trace()` no requiere imports adicionales porque opera sobre diccionarios nativos de Python.

```python
import json
```

In [ ]:
import json

## Caso 1 — Misclassification

**Problema:** el router envió la consulta al agente incorrecto.

**Sintoma en la traza:** `metadata.expected_intent != metadata.actual_intent`

**Escenario:** un usuario pregunta por su factura (`expected_intent: 'finance'`), pero el sistema lo clasifica como IT y responde con sugerencias tecnicas irrelevantes.

```
metadata: {
    'expected_intent': 'finance',   # lo que DEBERIA ser
    'actual_intent': 'it'           # lo que el router DIO
}
```

In [ ]:
trace_misclassification = {
    'trace_name': 'support-request',
    'input': {'query': 'No puedo ver mi factura'},
    'metadata': {
        'expected_intent': 'finance',
        'actual_intent': 'it'
    },
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'intent': 'it'},
            'duration_ms': 140
        },
        {
            'name': 'it-agent',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'response': 'Proba reiniciar la app.'},
            'duration_ms': 900
        }
    ],
    'output': {'final_response': 'Proba reiniciar la app.'}
}

trace_misclassification

## Caso 2 — Retrieval vacio

**Problema:** el sistema no encontro documentos relevantes en la base de conocimiento.

**Sintoma en la traza:** span de retrieval con `output['count'] == 0` o `output['documents'] == []`

**Escenario:** un usuario pregunta por politica de licencia por maternidad, el router detecta HR correctamente, pero el retrieval devuelve 0 documentos porque no hay data indexada sobre ese tema.

```
retrieval span: {
    'output': {
        'documents': [],    # no se encontro nada
        'count': 0          # cero resultados
    }
}
```

In [ ]:
trace_empty_retrieval = {
    'trace_name': 'rag-support-request',
    'input': {'query': 'Cual es la politica de licencia por maternidad?'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'Cual es la politica de licencia por maternidad?'},
            'output': {'intent': 'hr'},
            'duration_ms': 110
        },
        {
            'name': 'retrieval',
            'input': {'query': 'Cual es la politica de licencia por maternidad?'},
            'output': {'documents': [], 'count': 0},
            'duration_ms': 250
        },
        {
            'name': 'hr-agent',
            'input': {'query': 'Cual es la politica de licencia por maternidad?', 'context': ''},
            'output': {'response': 'No tengo informacion disponible sobre ese tema.'},
            'duration_ms': 820
        }
    ],
    'output': {'final_response': 'No tengo informacion disponible sobre ese tema.'}
}

trace_empty_retrieval

## Caso 3 — Latencia alta

**Problema:** un paso del sistema tarda mucho mas de lo esperado.

**Sintoma en la traza:** `duration_ms > 3000` en algun span

**Escenario:** el agente HR tardo 5.8 segundos en responder. El umbral de alerta se define en 3 segundos. Posibles causas: timeout de API, LLM lento, query compleja.

```
hr-agent span: {
    'duration_ms': 5800   # > 3000 -> ALERTA
}
```

In [ ]:
trace_high_latency = {
    'trace_name': 'support-request',
    'input': {'query': 'Como solicito vacaciones?'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'Como solicito vacaciones?'},
            'output': {'intent': 'hr'},
            'duration_ms': 95
        },
        {
            'name': 'hr-agent',
            'input': {'query': 'Como solicito vacaciones?'},
            'output': {'response': 'Para solicitar vacaciones ingresa al portal de RRHH.'},
            'duration_ms': 5800
        }
    ],
    'output': {'final_response': 'Para solicitar vacaciones ingresa al portal de RRHH.'}
}

trace_high_latency

## Caso 4 — Loop de agentes

**Problema:** dos o mas agentes se pasan la solicitud entre si sin llegar a una respuesta final.

**Sintoma en la traza:** el mismo `name` de span aparece mas de 2 veces

**Escenario:** un usuario tiene un problema mixto (laptop lenta + factura). IT-agent deriva a Finance, Finance deriva de vuelta a IT, y asi sucesivamente.

```
supervisor -> it-agent -> finance-agent -> it-agent -> finance-agent
                                                     ^--- loop detectado: 'it-agent' aparece 2 veces
```

In [ ]:
trace_loop = {
    'trace_name': 'support-request',
    'input': {'query': 'Mi laptop esta lenta y tengo un problema con mi factura'},
    'spans': [
        {'name': 'supervisor', 'output': {'next': 'it-agent'}, 'duration_ms': 130},
        {'name': 'it-agent', 'output': {'response': 'Revisa tu conexion', 'handoff': 'finance-agent'}, 'duration_ms': 720},
        {'name': 'finance-agent', 'output': {'response': 'Revisa tu factura', 'handoff': 'it-agent'}, 'duration_ms': 680},
        {'name': 'it-agent', 'output': {'response': 'Revisa tu conexion', 'handoff': 'finance-agent'}, 'duration_ms': 710},
        {'name': 'finance-agent', 'output': {'response': 'Revisa tu factura', 'handoff': None}, 'duration_ms': 690}
    ],
    'output': {'final_response': 'Revisa tu factura'}
}

trace_loop

## Caso 5 — Error silencioso

**Problema:** un span termina sin output y sin marcar error.

**Sintoma en la traza:** span con `output == None`

**Escenario:** el legal-agent ejecuta pero no produce respuesta. La duracion de 15ms sugiere que fallo inmediatamente (un LLM normal tardaria 500-2000ms). El sistema no registro ningun error.

```
legal-agent span: {
    'output': None,       # deberia ser un string
    'duration_ms': 15     #异常mente bajo -> sugiere crash temprano
}
```

In [ ]:
trace_silent_error = {
    'trace_name': 'support-request',
    'input': {'query': 'Necesito ver mi contrato'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'Necesito ver mi contrato'},
            'output': {'intent': 'legal'},
            'duration_ms': 100
        },
        {
            'name': 'legal-agent',
            'input': {'query': 'Necesito ver mi contrato'},
            'output': None,
            'duration_ms': 15
        }
    ],
    'output': {'final_response': None}
}

trace_silent_error

## TODO — Funcion `diagnose_trace`

Implementa la funcion que detecta automaticamente el tipo de problema en una traza.

### Orden de deteccion

1. **Misclassification** -> revisar `metadata`
2. **Retrieval vacio** -> revisar spans con 'retrieval' en el nombre
3. **Latencia alta** -> revisar `duration_ms` de cada span
4. **Loop de agentes** -> contar frecuencias de nombres de spans
5. **Error silencioso** -> revisar spans con `output == None`

### Formato de retorno

```python
{
    'problem': 'misclassification',     # o 'empty_retrieval', 'high_latency', 'agent_loop', 'silent_error', 'none'
    'details': 'expected_intent=finance vs actual_intent=it',  # explicacion del problema
    'suggested_fix': 'Revisar reglas de routing...',            # accion correctiva
}
```

In [ ]:
def diagnose_trace(trace: dict) -> dict:
    """
    Analiza una traza y devuelve el diagnostico del problema encontrado.

    Debe detectar (en orden):
    1. misclassification: metadata.expected_intent != metadata.actual_intent
    2. empty_retrieval: algun span de retrieval con output.count == 0
    3. high_latency: algun span con duration_ms > 3000
    4. agent_loop: algun nodo aparece mas de 2 veces en los spans
    5. silent_error: algun span con output == None

    Returns:
        dict con 'problem', 'details' y 'suggested_fix'
    """
    # TODO 1: detectar misclassification
    # Pista: comparar metadata.get('expected_intent') con metadata.get('actual_intent')

    # TODO 2: detectar retrieval vacio
    # Pista: buscar span con 'retrieval' en el nombre y output.get('count') == 0

    # TODO 3: detectar latencia alta
    # Pista: iterar spans y verificar duration_ms > 3000

    # TODO 4: detectar loop de agentes
    # Pista: Counter de span names, buscar si alguno aparece > 2 veces

    # TODO 5: detectar error silencioso
    # Pista: buscar span con output == None

    return {'problem': 'none', 'details': 'No se detectaron problemas.', 'suggested_fix': None}

print('Funcion definida.')

## Ejecutar el diagnostico en todos los casos

Probamos `diagnose_trace()` contra cada uno de los 5 casos y vemos los resultados.

In [ ]:
cases = [
    ('Misclassification', trace_misclassification),
    ('Retrieval vacio',   trace_empty_retrieval),
    ('Latencia alta',     trace_high_latency),
    ('Loop de agentes',  trace_loop),
    ('Error silencioso',  trace_silent_error)
]

for name, trace in cases:
    result = diagnose_trace(trace)
    print(f'--- {name} ---')
    print(f"  Problema:   {result.get('problem')}")
    print(f"  Detalles:   {result.get('details')}")
    print(f"  Fix sugerido: {result.get('suggested_fix')}")
    print()

In [ ]:
assert diagnose_trace(trace_misclassification)['problem'] == 'misclassification'
assert diagnose_trace(trace_empty_retrieval)['problem'] == 'empty_retrieval'
assert diagnose_trace(trace_high_latency)['problem'] == 'high_latency'
assert diagnose_trace(trace_loop)['problem'] == 'agent_loop'
assert diagnose_trace(trace_silent_error)['problem'] == 'silent_error'
print('Checks E03 OK')

## Errores comunes

| Error | Causa | Como detectarlo |
|---|---|---|
| No revisar `metadata` primero | Empezar por otros chequeos y no detectar misclassification | El problema reportado no es el mas grave |
| Usar `trace['metadata']` sin `.get()` | La clave puede no existir y lanza `KeyError` | Usar `trace.get('metadata', {})` como safe access |
| Confundir loop con repeticion normal | Un agente puede aparecer 2 veces si el usuario consulta dos temas | Definir umbral: > 2 repeticiones = loop |
| No detectar error silencioso por `output=None` | Revisar `output == None` en vez de `output is None` | La duracion anormalmente baja es otra pista |
| Retornar fix generico | Sugerir lo mismo para todo tipo de falla | Cada problema necesita su propia accion correctiva |

## Sintesis

### Los 5 patrones de falla

| Tipo | Como se detecta | Accion correctiva tipica |
|---|---|---|
| Misclassification | `expected_intent != actual_intent` | Agregar reglas al router o mejorar prompts |
| Retrieval vacio | Span con `count == 0` | Indexar mas documentos o mejorar chunking |
| Latencia alta | `duration_ms > 3000` | Optimizar el paso lento o aumentar timeout |
| Loop de agentes | Mismo nombre > 2 veces | Agregar max_handoffs o mejorar logica de delegacion |
| Error silencioso | `output == None` | Agregar try/except y logging de errores |

### Relacion con otros ejercicios

| Ejercicio | Conexion con E03 |
|---|---|
| **E04** | Golden datasets para prevenir misclassification antes de producirla |
| **E05** | Router v1 vs v2: comparar accuracy entre versiones |
| **E11** | Alertas automaticas cuando se detectan estos patrones en produccion |
| **E12** | Ciclo de mejora: diagnosticar -> corregir -> medir de nuevo |